# 06 — Barycenter Averaging

This notebook walks through the symbolic barycenter estimation pipeline (Chapter 6).

**Pipeline overview:**
1. **TWE Distance** — Compute pairwise Temporal-Wasserstein TWE distances between symbolic sequences
2. **Edit-Shape DTW** — Align sequences using Edit-Shape DTW with rTWE as inner cost
3. **Pairwise Distance Matrix** — Full cohort distance matrix with group-level visualizations
4. **DBA Barycenter** — DTW Barycenter Averaging per diagnosis group (Control / TBI / RIL)
5. **Alignment Visualization** — Alignment paths with operation-colored arrows
6. **Hierarchical Community Detection** — Transition matrices, tree construction, community grouping

**Key dependencies:** `aeon` (time-series distances, only on HPC), `networkx` (graph/tree operations).

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from IPython.display import display

# --- aeon (HPC only) ---
try:
    from aeon.clustering.averaging import elastic_barycenter_average
    from aeon.distances import (
        twe_alignment_path,
        twe_cost_matrix,
        twe_pairwise_distance,
    )
    from aeon.distances._rtwe import (
        rtwe_alignment_path,
        rtwe_alignment_path_with_costs,
        rtwe_distance,
    )
    from aeon.distances._twe import _twe_cost_matrix

    HAS_AEON = True
except ImportError:
    HAS_AEON = False
    print("aeon not available — distance / barycenter cells will be skipped.")

# --- smartflat: config & data loading ---
from smartflat.configs.loader import import_config
from smartflat.constants import incomplete_clinical_administrations
from smartflat.features.symbolization.main import build_clusterdf
from smartflat.features.symbolization.utils import fix_clinical_diagnosis, get_prototypes
from smartflat.features.symbolization.utils_dataset import get_experiments_dataframe
from smartflat.features.symbolization.co_clustering import (
    compute_multimodal_matrices,
    append_foreground_background_distance,
)

# --- smartflat: symbolic barycenter ---
from smartflat.features.symbolic_barycenter.main import didactic_eshape_dtw_cost_matrix
from smartflat.features.symbolic_barycenter.visualization import (
    plot_chronogram_alignment,
    plot_distance_violin,
    plot_pairwise_twe_distances_by_group,
    plot_rtwe,
    plot_shapes_signal,
    plot_signals,
)
from smartflat.features.symbolic_barycenter.hierarchical_states_transitions import (
    create_cohort_community_bag,
    draw_tree,
    get_adjacency_matrix,
    get_motif_usage,
    graph_to_tree,
)

# --- smartflat: utilities ---
from smartflat.utils.utils import get_upsampled_labels
from smartflat.utils.utils_io import get_data_root
from smartflat.utils.utils_visualization import plot_chronogames

In [ ]:
# --- Experiment parameters ---
annotator_id = 'samperochon'
round_number = 8
symbolization_config_name = 'SymbolicSourceInferenceGoldConfig'
config = import_config(symbolization_config_name)
clustering_config_name = config.clustering_config_name

# TWE / rTWE parameters
nu = 0.001
lmbda = 1.0
n_samples = 200
upsampling = 'interpolation'
labels_col = 'symb_labels'

data_root = get_data_root()
print(f"Data root: {data_root}")
print(f"Clustering config: {clustering_config_name}")

In [ ]:
# Load experiment dataframe with symbolic labels
df = get_experiments_dataframe(
    experiment_config_name=symbolization_config_name,
    annotator_id=annotator_id,
    round_number=round_number,
    return_symbolization=True,
    return_data_only=True,
)
print(f"Loaded {len(df)} recordings")

# Clinical filtering
df = fix_clinical_diagnosis(df)
df = df[~df['participant_id'].isin(incomplete_clinical_administrations.keys())]
df.sort_values('task_number_int', ascending=True, inplace=True)
df = df[df['pathologie'].isin(['HEALTHY', 'RIL', 'TBI'])]
df.drop_duplicates(subset=['trigram'], keep='first', inplace=True)

print(f"After filtering: {len(df)} recordings")
display(df.groupby('pathologie')['participant_id'].nunique().to_frame('n_participants'))

## 1. Temporal-Wasserstein TWE Distance

Compute pairwise TW-TWE distances between symbolic sequences, using
Wasserstein distance between symbols' temporal occurrence distributions.

See: `smartflat.features.symbolic_barycenter.main`

In [ ]:
# Upsample symbolic label sequences to uniform length for pairwise distance computation
X = get_upsampled_labels(
    df, labels_col=labels_col, use_shortest=False,
    n_samples=n_samples, upsampling=upsampling,
)
print(f"Upsampled matrix: {X.shape}  (n_sequences x max_duration)")
print(f"Unique symbols in cohort: {len(np.unique(X))}")

In [ ]:
# Precomputed prototype distance matrix (G-space) — used as ground cost for TWE/rTWE
_, _, D_GF = compute_multimodal_matrices(
    symbolization_config_name,
    annotator_id=annotator_id,
    round_number=round_number,
    input_space='K_space',
    verbose=False,
)
D_G = append_foreground_background_distance(D_GF, method='max', alpha=0.0)

print(f"Prototype distance matrix D_G: {D_G.shape}")
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(D_G, cmap='viridis', ax=ax)
ax.set_title('Prototype distance matrix (G-space)')
plt.tight_layout()
plt.show()

In [ ]:
if HAS_AEON:
    # Pairwise TWE distance matrix across all upsampled sequences
    # Each sequence is cast to float64 2D array as required by aeon
    X_aeon = X.astype(np.float64)
    if X_aeon.ndim == 1:
        X_aeon = X_aeon[:, np.newaxis]
    if X_aeon.ndim == 2:
        X_aeon = X_aeon[:, np.newaxis, :]  # (n, 1, T) for univariate

    D_twe = twe_pairwise_distance(X_aeon, nu=nu, lmbda=lmbda)
    print(f"Pairwise TWE distance matrix: {D_twe.shape}")
    print(f"Distance range: [{D_twe[D_twe > 0].min():.4f}, {D_twe.max():.4f}]")

    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(D_twe, cmap='viridis', ax=ax)
    ax.set_title(f'Pairwise TWE distances (nu={nu}, lmbda={lmbda})')
    plt.tight_layout()
    plt.show()
else:
    print("Skipped — aeon not available")

## 2. Edit-Shape DTW with rTWE

Align symbolic sequences using the Edit-Shape DTW algorithm with
registered Time Warp Edit (rTWE) as the inner cost function.

See: `smartflat.features.symbolic_barycenter.main.didactic_eshape_dtw_cost_matrix`

In [ ]:
if HAS_AEON:
    # Pick a sample pair of sequences for didactic visualization
    idx_a, idx_b = 0, 1
    x = X_aeon[idx_a, 0, :]
    y = X_aeon[idx_b, 0, :]
    print(f"Sequence A (idx={idx_a}): length={len(x)}, participant={df.iloc[idx_a]['participant_id']}")
    print(f"Sequence B (idx={idx_b}): length={len(y)}, participant={df.iloc[idx_b]['participant_id']}")

    # Didactic Edit-Shape DTW with rTWE inner distance
    didactic_eshape_dtw_cost_matrix(
        x, y,
        nu=nu, lmbda=lmbda,
        precomputed_distances=D_G,
        step_sequ=1,
        lmbda_rtwe=0.1, nu_rtwe=0.01,
        n_plot_max=5,
        verbose=True,
    )
else:
    print("Skipped — aeon not available")

In [ ]:
if HAS_AEON:
    # Side-by-side comparison: TWE vs rTWE cost matrices for the same pair
    cost_twe = twe_cost_matrix(x, y, nu=nu, lmbda=lmbda)
    cost_rtwe = _twe_cost_matrix(x, y, nu=nu, lmbda=lmbda)

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, cost, title in zip(axes, [cost_twe, cost_rtwe], ['TWE cost matrix', 'rTWE cost matrix']):
        im = ax.imshow(cost, aspect='auto', cmap='coolwarm', origin='lower')
        ax.set_title(title)
        ax.set_xlabel('Sequence B')
        ax.set_ylabel('Sequence A')
        plt.colorbar(im, ax=ax, fraction=0.046)
    plt.suptitle(f'Cost matrix comparison (nu={nu}, lmbda={lmbda})')
    plt.tight_layout()
    plt.show()
else:
    print("Skipped — aeon not available")

## 3. Pairwise Distance Matrix

Build the full pairwise distance matrix across all participants and
visualize group-level distance distributions (Control vs TBI vs RIL).

See: `smartflat.features.symbolic_barycenter.visualization.plot_pairwise_twe_distances_by_group`

In [ ]:
if HAS_AEON:
    # Full pairwise TWE distance matrix (computed above; reuse D_twe)
    # Sort rows/cols by pathologie for visual grouping
    patho_order = df['pathologie'].values
    sort_idx = np.argsort(patho_order)
    D_sorted = D_twe[sort_idx][:, sort_idx]
    patho_sorted = patho_order[sort_idx]

    fig, ax = plt.subplots(figsize=(9, 8))
    sns.heatmap(D_sorted, cmap='viridis', ax=ax)

    # Add group separators
    boundaries = np.where(np.diff(patho_sorted.astype(str) != np.roll(patho_sorted.astype(str), 1)))[0]
    for b in boundaries:
        ax.axhline(b, color='white', linewidth=0.8)
        ax.axvline(b, color='white', linewidth=0.8)

    ax.set_title(f'Pairwise TWE distances sorted by pathology (nu={nu}, lmbda={lmbda})')
    plt.tight_layout()
    plt.show()
else:
    print("Skipped — aeon not available")

In [ ]:
if HAS_AEON:
    # KDE curves and heatmap of intra-group TWE distances
    plot_pairwise_twe_distances_by_group(D_twe, df, covar_col='pathologie')
    plt.show()
else:
    print("Skipped — aeon not available")

In [ ]:
if HAS_AEON:
    # Interactive violin plot: intra- vs inter-group distance distributions
    plot_distance_violin(df, D_twe, covar_col='pathologie')
else:
    print("Skipped — aeon not available")

## 4. DBA Barycenter Estimation

Compute representative "average sequences" (barycenters) per diagnosis
group using adapted DTW Barycenter Averaging (DBA) under TW-TWE distance.

Key parameters: train/test split (50%), convergence in <20 iterations.

In [ ]:
if HAS_AEON:
    # Train/test split: 50% stratified by pathologie
    from sklearn.model_selection import train_test_split

    train_idx, test_idx = train_test_split(
        np.arange(len(df)), test_size=0.5, stratify=df['pathologie'].values, random_state=42,
    )
    df_train = df.iloc[train_idx].copy()
    df_test = df.iloc[test_idx].copy()
    X_train = X_aeon[train_idx]
    X_test = X_aeon[test_idx]

    print(f"Train: {len(df_train)}  |  Test: {len(df_test)}")
    for patho in ['HEALTHY', 'RIL', 'TBI']:
        n_tr = (df_train['pathologie'] == patho).sum()
        n_te = (df_test['pathologie'] == patho).sum()
        print(f"  {patho}: train={n_tr}, test={n_te}")
else:
    print("Skipped — aeon not available")

In [ ]:
if HAS_AEON:
    # Compute DBA barycenter per pathology group on the training set
    barycenters = {}
    for patho in ['HEALTHY', 'RIL', 'TBI']:
        mask = df_train['pathologie'].values == patho
        X_group = X_train[mask]
        print(f"\n--- {patho} ({X_group.shape[0]} sequences) ---")

        barycenter = elastic_barycenter_average(
            X_group,
            method='petitjean',
            distance='twe',
            nu=nu, lmbda=lmbda,
            init_barycenter='random',
            max_iters=100,
            tol=1e-5,
        )
        barycenters[patho] = barycenter
        print(f"  Barycenter shape: {barycenter.shape}")
        print(f"  Unique symbols in barycenter: {len(np.unique(barycenter))}")
else:
    print("Skipped — aeon not available")

In [ ]:
if HAS_AEON:
    # Visualize barycenters as chronograms — one per pathology group
    colors = {'HEALTHY': 'tab:blue', 'RIL': 'tab:red', 'TBI': 'tab:orange'}

    for patho, barycenter in barycenters.items():
        b = barycenter.squeeze()
        plot_signals(b, b, title=f'{patho} barycenter chronogram')
        plt.show()
else:
    print("Skipped — aeon not available")

In [ ]:
if HAS_AEON:
    # Convergence analysis: iteratively compute barycenter and track cost
    # Run on HEALTHY group as example (smaller, faster)
    mask_h = df_train['pathologie'].values == 'HEALTHY'
    X_h = X_train[mask_h]

    costs_per_iter = []
    barycenter_iter = None
    n_iters = 20

    for i in range(n_iters):
        barycenter_iter = elastic_barycenter_average(
            X_h,
            method='petitjean',
            distance='twe',
            nu=nu, lmbda=lmbda,
            init_barycenter=barycenter_iter if i > 0 else 'random',
            max_iters=1,
            tol=0,
        )
        # Sum of TWE distances from each sequence to the barycenter
        b = barycenter_iter.squeeze()
        total_cost = sum(
            twe_cost_matrix(b, X_h[j, 0, :], nu=nu, lmbda=lmbda)[-1, -1]
            for j in range(X_h.shape[0])
        )
        costs_per_iter.append(total_cost)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(range(1, n_iters + 1), costs_per_iter, 'o-', color='tab:blue')
    ax.set(xlabel='DBA iteration', ylabel='Total TWE cost',
           title='DBA convergence (HEALTHY group)')
    plt.tight_layout()
    plt.show()
else:
    print("Skipped — aeon not available")

## 5. Alignment Visualization

Visualize alignment paths between sequences and their barycenters,
with colored arrows for operation types (match, insertion, deletion).

See: `smartflat.features.symbolic_barycenter.visualization.plot_chronogram_alignment`

In [ ]:
if HAS_AEON:
    # Compute alignment path between the HEALTHY barycenter and a sample sequence
    b_healthy = barycenters['HEALTHY'].squeeze()
    sample_idx = np.where(df_train['pathologie'].values == 'HEALTHY')[0][0]
    x_sample = X_train[sample_idx, 0, :]

    alignment_path = twe_alignment_path(b_healthy, x_sample, nu=nu, lmbda=lmbda)
    print(f"Alignment path length: {len(alignment_path)}")
    print(f"Barycenter length: {len(b_healthy)}, Sequence length: {len(x_sample)}")
else:
    print("Skipped — aeon not available")

In [ ]:
if HAS_AEON:
    # Chronogram alignment: colored arrows for match, insertion, deletion, mismatch
    plot_chronogram_alignment(
        b_healthy, x_sample,
        paths=alignment_path,
        nu=nu, lmbda=lmbda,
        method='twe',
        title=f'HEALTHY barycenter vs. participant {df_train.iloc[sample_idx]["participant_id"]}',
    )
    plt.show()
else:
    print("Skipped — aeon not available")

In [ ]:
if HAS_AEON:
    # rTWE cost decomposition: match-same, match-previous, delete-x, delete-y
    alignment_with_costs = rtwe_alignment_path_with_costs(b_healthy, x_sample, nu=nu, lmbda=lmbda)

    # Unpack costs from alignment (implementation-dependent structure)
    if isinstance(alignment_with_costs, tuple) and len(alignment_with_costs) >= 6:
        match_same_costs, match_prev_costs, del_x_costs, del_y_costs, total_costs, option_paths = (
            alignment_with_costs[0], alignment_with_costs[1],
            alignment_with_costs[2], alignment_with_costs[3],
            alignment_with_costs[4], alignment_with_costs[5],
        )
        plot_rtwe(
            match_same_costs, match_prev_costs,
            del_x_costs, del_y_costs,
            total_costs, option_paths,
            title='rTWE cost decomposition (HEALTHY barycenter vs. sample)',
        )
        plt.show()
    else:
        print(f"rtwe_alignment_path_with_costs returned {type(alignment_with_costs)} — "
              "check aeon version for cost decomposition API")
else:
    print("Skipped — aeon not available")

## 6. Hierarchical Community Detection

Build hierarchical trees from transition matrices and detect communities
via tree-cutting for motif grouping.

See: `smartflat.features.symbolic_barycenter.hierarchical_states_transitions`

In [ ]:
# Load prototypes and build cluster-level statistics
P = get_prototypes(
    clustering_config_name,
    annotator_id=annotator_id,
    round_number=round_number,
    verbose=True,
)
n_clusters = len(P)
print(f"Prototypes: {n_clusters}")

clusterdf = build_clusterdf(
    df,
    clustering_config_name=symbolization_config_name,
    embeddings_label_col='symb_labels',
    segments_labels_col='symb_segments_labels',
    segments_length_col='symb_segments_length',
    annotator_id=annotator_id,
    round_number=round_number,
)
print(f"Cluster DataFrame: {len(clusterdf)} clusters")
display(clusterdf[['cluster_index', 'cluster_type', 'n_subjects', 'n_appearance']].head(10))

In [ ]:
# Extract cohort-level motif labels from symbolic sequences
cohort_motif_labels = np.concatenate(df[labels_col].dropna().values)
print(f"Total motif labels: {len(cohort_motif_labels)}")
print(f"Unique motifs: {len(np.unique(cohort_motif_labels))}")

# Compute adjacency, transition, and temporal matrices
adjacency_matrix, transition_matrix, temporal_matrix = get_adjacency_matrix(
    cohort_motif_labels, n_clusters,
)
print(f"Adjacency matrix: {adjacency_matrix.shape}")
print(f"Transition matrix: {transition_matrix.shape}")
print(f"Temporal matrix: {temporal_matrix.shape}")

In [ ]:
# Visualize adjacency, transition, and temporal matrices side by side
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
matrices = {
    'Adjacency (raw counts)': adjacency_matrix,
    'Transition (row-normalized)': transition_matrix,
    'Temporal (mean lag)': temporal_matrix,
}
for ax, (title, matrix) in zip(axes, matrices.items()):
    sns.heatmap(matrix, cmap='coolwarm', ax=ax, square=True)
    ax.set_title(title)
    ax.set_xlabel('To')
    ax.set_ylabel('From')

plt.suptitle('State transition analysis', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Motif usage: count how often each prototype appears across the cohort
motif_usage = get_motif_usage(cohort_motif_labels, n_clusters)
print(f"Motif usage vector: {motif_usage.shape}")
print(f"Top-10 most frequent motifs:")
top_idx = np.argsort(motif_usage)[::-1][:10]
for i in top_idx:
    print(f"  Motif {i}: {motif_usage[i]} occurrences")

fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(range(n_clusters), motif_usage, color='steelblue')
ax.set(xlabel='Motif index', ylabel='Count', title='Motif usage across cohort')
plt.tight_layout()
plt.show()

In [ ]:
# Build hierarchical tree from transition matrix via agglomerative merging
motif_usage_dict = {i: motif_usage[i] for i in range(n_clusters)}
T = graph_to_tree(motif_usage, transition_matrix, n_clusters, merge_sel=1)
print(f"Tree nodes: {T.number_of_nodes()}, edges: {T.number_of_edges()}")

In [ ]:
# Visualize the hierarchical tree — node size scaled by motif usage
draw_tree(T, fig_width=20.0, usage_dict=motif_usage_dict, save_to_file=False, show_figure=True)

In [ ]:
# Community detection via tree-cutting: group motifs into communities
cohort_community_bag = create_cohort_community_bag(
    motif_labels=cohort_motif_labels,
    trans_mat_full=transition_matrix,
    cut_tree=None,  # automatic cut
    n_clusters=n_clusters,
    segmentation_algorithm='kmeans',
)
print(f"Communities detected: {len(set(cohort_community_bag))}")
for c in sorted(set(cohort_community_bag)):
    n = (np.array(cohort_community_bag) == c).sum()
    print(f"  Community {c}: {n} motif assignments")

In [ ]:
# Map motif labels to community labels per participant and visualize
community_bag_array = np.array(cohort_community_bag)

# Build a motif→community mapping from the bag (index = motif, value = community)
motif_to_community = np.zeros(n_clusters, dtype=int)
for motif_idx in range(n_clusters):
    mask = cohort_motif_labels == motif_idx
    if mask.any():
        motif_to_community[motif_idx] = int(np.median(community_bag_array[mask]))

# Apply mapping to each participant's symbolic sequence
df['community_labels'] = df[labels_col].apply(
    lambda seq: np.array([motif_to_community[int(s)] if int(s) < n_clusters else -1 for s in seq])
    if seq is not None else None
)

plot_chronogames(df, labels_col='community_labels', n_subjects=12, upsampling=upsampling)
plt.suptitle('Community-level symbolic sequences', y=1.02)
plt.show()